<a href="https://colab.research.google.com/github/arthur-gui-22/PROJETOSUSTENT-VEL/blob/CODE-HTML/estatistica_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Atividade de Estatística em Python

# 1) Preparação do ambiente

In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_diabetes

random.seed(42)
np.random.seed(42)

#2) Carregar o dataset Wine

In [ ]:
wine = load_wine(as_frame=True)   # Carrega o dataset já como DataFrame
df = wine.frame.copy()            # Cria uma cópia para manipulação

# Renomear a coluna alvo de 'target' para 'class'
df = df.rename(columns={"target": "class"})

# Criar um dicionário com os nomes das classes (0, 1, 2 → Cultivares)
class_names = {i: name for i, name in enumerate(wine.target_names)}

# Adicionar coluna com os nomes das classes
df["class_name"] = df["class"].map(class_names)

# Mostrar dimensões e primeiras linhas
print("Dimensões do dataset:", df.shape)
df.head()


#3) Exploração inicial e estatística descritiva

In [ ]:
# Primeiras linhas
print("Primeiras linhas do dataset:")
print(df.head())

# Informações sobre tipos de dados e memória
print("\nInformações do dataset:")
print(df.info())

# Estatísticas resumo das variáveis numéricas
print("\nEstatísticas descritivas:")
print(df.describe())

# Checagem de valores ausentes (NA)
print("\nValores ausentes por coluna:")
print(df.isnull().sum())

# 4) Medidas de tendência central

In [ ]:
# Selecionar apenas colunas numéricas contínuas
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
# Remover a coluna 'class' do conjunto de numéricas
numericas = [c for c in numericas if c != "class"]

# --- Medidas globais ---
mean_vals   = df[numericas].mean()
median_vals = df[numericas].median()
mode_vals   = df[numericas].mode().iloc[0]  # se houver múltiplas modas, pega a primeira

print("Médias globais:")
display(mean_vals.to_frame("mean"))

print("Medianas globais:")
display(median_vals.to_frame("median"))

print("Modas globais (primeira ocorrência):")
display(mode_vals.to_frame("mode"))

# --- Medidas por classe ---
grouped = df.groupby("class")[numericas]

print("\nMédias por classe:")
display(grouped.mean())

print("\nMedianas por classe:")
display(grouped.median())

print("\nModas por classe (primeira ocorrência por coluna):")
modes_by_class = grouped.apply(lambda g: g.mode().iloc[0])
display(modes_by_class)

#5) Medidas de dispersão

In [ ]:
# 5) Medidas de dispersão

# Selecionar apenas colunas numéricas contínuas
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
numericas = [c for c in numericas if c != "class"]

# --- Medidas globais ---
max_vals = df[numericas].max()
min_vals = df[numericas].min()
range_vals = max_vals - min_vals
var_vals   = df[numericas].var(ddof=1)   # variância amostral
std_vals   = df[numericas].std(ddof=1)   # desvio padrão amostral
q1         = df[numericas].quantile(0.25)
q3         = df[numericas].quantile(0.75)
iqr_vals   = q3 - q1

print("Amplitude (global):")
display(range_vals.to_frame("range"))

print("Variância (global):")
display(var_vals.to_frame("variance"))

print("Desvio Padrão (global):")
display(std_vals.to_frame("std"))

print("IQR (global):")
display(iqr_vals.to_frame("IQR"))

# --- Medidas por classe ---
def dispersion_summary(g):
    out = {}
    out["range"]    = g.max() - g.min()
    out["variance"] = g.var(ddof=1)
    out["std"]      = g.std(ddof=1)
    out["IQR"]      = g.quantile(0.75) - g.quantile(0.25)
    return pd.concat(out, axis=1)

disp_by_class = df.groupby("class")[numericas].apply(dispersion_summary)

print("\nMedidas de dispersão por classe:")
display(disp_by_class)


#6) Histogramas (distribuição)

In [ ]:
# 6) Histogramas (distribuição)
# Visualizar a distribuição de variáveis numéricas com histogramas

for col in numericas:
    plt.figure()
    plt.hist(df[col].dropna(), bins=20)  # 20 intervalos (bins)
    plt.title(f"Histograma - {col}")
    plt.xlabel(col)
    plt.ylabel("Frequência")
    plt.show()


#7) Boxplots (dispersão e outliers) por classe

In [ ]:
# 7) Boxplots (dispersão e outliers) por classe

# Identificar os rótulos das classes
class_labels = sorted(df["class"].unique().tolist())

# Criar boxplots para cada variável numérica
for col in numericas:
    # Agrupar os dados da variável por classe
    grouped_data = [df.loc[df["class"] == cl, col].dropna().values for cl in class_labels]

    # Criar o boxplot
    plt.figure()
    plt.boxplot(grouped_data, labels=[str(c) for c in class_labels], showmeans=True)
    plt.title(f"Boxplot por classe - {col}")
    plt.xlabel("Classe (0, 1, 2)")
    plt.ylabel(col)
    plt.show()


#8) Identificação simples de outliers (regra do IQR)

In [ ]:

# Um valor é considerado outlier se estiver abaixo de Q1 - 1.5*IQR ou acima de Q3 + 1.5*IQR.

def iqr_outlier_mask(s: pd.Series, factor: float = 1.5) -> pd.Series:
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return (s < lower) | (s > upper)

# Dicionário para armazenar relatório de outliers
outlier_report = {}
for col in numericas:
    mask = iqr_outlier_mask(df[col])
    outlier_report[col] = {
        "n_outliers": int(mask.sum()),        # número de outliers
        "idx": df.index[mask].tolist()        # índices dos outliers
    }

# Converter para DataFrame para visualização
outlier_df = pd.DataFrame(outlier_report)
display(outlier_df)

#9) Conclusões rápidas (exemplo orientativo)

In [ ]:
# 1) Preparação
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

In [ ]:

# Carregar dataset Diabetes
diabetes = load_diabetes()
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df["target"] = diabetes.target

In [ ]:
# 2) Exploração inicial
print("Primeiras linhas:")
print(df.head())

print("\nInformações do dataset:")
print(df.info())

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

print("\nEstatísticas descritivas:")
print(df.describe())

In [ ]:
# 3) Medidas de tendência central (globais)
mean_vals   = df.mean(numeric_only=True)
median_vals = df.median(numeric_only=True)
mode_vals   = df.mode(numeric_only=True).iloc[0]

print("\nMédias globais:")
display(mean_vals.to_frame("mean"))

print("\nMedianas globais:")
display(median_vals.to_frame("median"))

print("\nModas globais:")
display(mode_vals.to_frame("mode"))

In [ ]:
# 4) Medidas de dispersão (globais)
range_vals = df.max(numeric_only=True) - df.min(numeric_only=True)
var_vals   = df.var(numeric_only=True, ddof=1)
std_vals   = df.std(numeric_only=True, ddof=1)
iqr_vals   = df.quantile(0.75, numeric_only=True) - df.quantile(0.25, numeric_only=True)

print("\nAmplitude (range) global:")
display(range_vals.to_frame("range"))

print("\nVariância global:")
display(var_vals.to_frame("variance"))

print("\nDesvio padrão global:")
display(std_vals.to_frame("std"))

print("\nIQR global:")
display(iqr_vals.to_frame("IQR"))


In [ ]:
# 5) Histogramas
for col in df.columns:
    plt.figure()
    plt.hist(df[col].dropna(), bins=20)
    plt.title(f"Histograma - {col}")
    plt.xlabel(col)
    plt.ylabel("Frequência")
    plt.show()

In [ ]:

# 6) Boxplots
for col in df.columns:
    plt.figure()
    plt.boxplot(df[col].dropna(), showmeans=True)
    plt.title(f"Boxplot - {col}")
    plt.xlabel(col)
    plt.ylabel(col)